In [1]:
import os
import requests
import json
import pandas as pd
import xml.etree.ElementTree as ET
import logging
import time
import datetime
from secret_keys import PUBMED_API_KEY
#initiliase logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', filename='pubmed_fetch.log')

# Replace with your actual API key
API_KEY = PUBMED_API_KEY

def load_processed_ids(results_file):
    """Load processed IDs from the existing results file."""
    try:
        df_existing = pd.read_csv(results_file, usecols=['PMID'], dtype={'PMID': str})
        processed_ids = set(df_existing['PMID'].tolist())
        logging.info(f"Loaded {len(processed_ids)} processed IDs from {results_file}")
    except FileNotFoundError:
        processed_ids = set()
        logging.info(f"No existing results file found at {results_file}")
    except Exception as e:
        logging.error(f"Error loading processed IDs: {e}")
        processed_ids = set()
    return processed_ids

def get_count_for_date_range(query, mindate, maxdate):
    """Get the count of results for the specified date range."""
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    search_params = {
        'db': 'pubmed',
        'term': query,
        'datetype': 'pdat',
        'mindate': mindate,
        'maxdate': maxdate,
        'rettype': 'count',
        'retmode': 'json',
        'api_key': API_KEY
    }
    try:
        response = requests.get(f"{base_url}esearch.fcgi", params=search_params)
        response.raise_for_status()
        data = response.json()
        count = int(data['esearchresult']['count'])
        return count
    except requests.exceptions.RequestException as e:
        logging.error(f"Network error while fetching count for date range {mindate} to {maxdate}: {e}")
        return 0
    except ValueError as e:
        logging.error(f"JSON decoding error while fetching count for date range {mindate} to {maxdate}: {e}")
        logging.debug(f"Response content: {response.text}")
        return 0

def get_new_pubmed_ids(query, mindate, maxdate):
    """Retrieve PubMed IDs based on the search query and date range."""
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    retmax = 10000  # Max IDs per request
    new_ids = []

    search_params = {
        'db': 'pubmed',
        'term': query,
        'datetype': 'pdat',
        'mindate': mindate,
        'maxdate': maxdate,
        'retmode': 'json',
        'retmax': retmax,
        'api_key': API_KEY
    }

    try:
        response = requests.get(f"{base_url}esearch.fcgi", params=search_params)
        response.raise_for_status()
        data = response.json()
        total_count = int(data['esearchresult']['count'])
        logging.info(f"Date range {mindate} to {maxdate}: Found {total_count} IDs.")

        if total_count == 0:
            return []

        # Fetch IDs in batches if total_count exceeds retmax
        for retstart in range(0, total_count, retmax):
            search_params.update({'retstart': retstart})
            response = requests.get(f"{base_url}esearch.fcgi", params=search_params)
            response.raise_for_status()
            data = response.json()
            id_list = data['esearchresult']['idlist']
            new_ids.extend(id_list)
            logging.info(f"Retrieved {len(new_ids)}/{total_count} IDs.")
            time.sleep(0.1)  # Adjust sleep time as needed

    except requests.exceptions.RequestException as e:
        logging.error(f"Network error while fetching IDs for date range {mindate} to {maxdate}: {e}")
    except ValueError as e:
        logging.error(f"JSON decoding error while fetching IDs for date range {mindate} to {maxdate}: {e}")
        logging.debug(f"Response content: {response.text}")

    return new_ids

def fetch_pubmed_details_from_ids(id_list, processed_ids):
    """Fetch PubMed article details given a list of IDs, skipping processed IDs."""
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    batch_size = 200  # Max IDs per request
    total_ids = len(id_list)

    for start in range(0, total_ids, batch_size):
        batch_ids = id_list[start:start + batch_size]
        # Exclude IDs that have been processed
        batch_ids = [i for i in batch_ids if i not in processed_ids]
        if not batch_ids:
            continue

        fetch_params = {
            'db': 'pubmed',
            'id': ','.join(batch_ids),
            'retmode': 'xml',
            'api_key': API_KEY
        }

        try:
            response = requests.post(f"{base_url}efetch.fcgi", data=fetch_params, timeout=30)
            response.raise_for_status()
            batch_records = parse_pubmed_details(response.text)
            # Save records incrementally
            save_records_incrementally(batch_records, 'pubmed_results_complete.csv')
            processed_ids.update(batch_ids)
            logging.info(f"Fetched and saved details for batch starting at index {start}.")
            time.sleep(0.1)
        except requests.exceptions.RequestException as e:
            logging.error(f"Network error while fetching details for batch starting at index {start}: {e}")
        except ValueError as e:
            logging.error(f"JSON decoding error while fetching details for batch starting at index {start}: {e}")
            logging.debug(f"Response content: {response.text}")

def parse_pubmed_details(xml_data):
    """Parse XML data returned by efetch."""
    records = []
    root = ET.fromstring(xml_data)
    for article in root.findall(".//PubmedArticle"):
        record = {}
        record["PMID"] = article.findtext(".//PMID")
        record["Title"] = article.findtext(".//ArticleTitle")
        abstract = article.findtext(".//Abstract/AbstractText")
        record["Abstract"] = abstract if abstract else ""
        authors = []
        for author in article.findall(".//Author"):
            last_name = author.findtext("LastName")
            fore_name = author.findtext("ForeName")
            if last_name and fore_name:
                authors.append(f"{last_name} {fore_name}")
            elif last_name:
                authors.append(last_name)
            elif fore_name:
                authors.append(fore_name)
        record["Authors"] = ", ".join(authors)
        record["Date"] = article.findtext(".//PubDate/Year")
        record["Journal"] = article.findtext(".//Journal/Title")
        records.append(record)
    return records

def save_records_incrementally(records, results_file):
    """Append new records to the existing results CSV file."""
    df_new = pd.DataFrame(records)
    try:
        if os.path.isfile(results_file):
            df_new.to_csv(results_file, mode='a', index=False, header=False)
        else:
            df_new.to_csv(results_file, index=False)
        logging.info(f"Appended {len(df_new)} records to {results_file}")
    except Exception as e:
        logging.error(f"Error saving records to {results_file}: {e}")

def process_date_range(query, start_date, end_date, processed_ids, all_ids):
    """Process the date range by splitting it into smaller ranges as needed."""
    # Get the count for the date range
    mindate = start_date.strftime('%Y/%m/%d')
    maxdate = end_date.strftime('%Y/%m/%d')
    count = get_count_for_date_range(query, mindate, maxdate)
    logging.info(f"Date range {mindate} to {maxdate} - Count: {count}")

    if count <= 10000:
        # Fetch and process IDs for this date range
        new_ids = get_new_pubmed_ids(query, mindate, maxdate)
        new_ids_set = set(new_ids)
        # Update all_ids and processed_ids
        all_ids.update(new_ids_set)
        ids_to_fetch = new_ids_set - processed_ids
        logging.info(f"IDs to fetch for date range {mindate} to {maxdate}: {len(ids_to_fetch)}")
        if ids_to_fetch:
            fetch_pubmed_details_from_ids(list(ids_to_fetch), processed_ids)
    else:
        # Determine the level of splitting
        delta_days = (end_date - start_date).days
        logging.info(f"Date range {mindate} to {maxdate} - Splitting by {delta_days} days.")
        
        if delta_days > 365:  # Split by year
            logging.info(f"Splitting {mindate} to {maxdate} by year.")
            for year in range(start_date.year, end_date.year + 1):
                year_start = datetime.date(year, 1, 1)
                year_end = datetime.date(year, 12, 31)
                # Adjust dates if they are out of the original range
                if year_start < start_date:
                    year_start = start_date
                if year_end > end_date:
                    year_end = end_date
                logging.debug(f"Processing year range: {year_start} to {year_end}")
                process_date_range(query, year_start, year_end, processed_ids, all_ids)
        
        elif delta_days >= 28:  # Split by month
            logging.info(f"Splitting {mindate} to {maxdate} by month.")
            current_date = start_date
            while current_date <= end_date:
                month_start = current_date.replace(day=1)
                # Calculate the next month
                if current_date.month == 12:
                    year = current_date.year + 1
                    month = 1
                else:
                    year = current_date.year
                    month = current_date.month + 1
                try:
                    next_month = datetime.date(year, month, 1)
                except ValueError:
                    # Handle invalid dates
                    logging.error(f"Invalid date for next month: Year={year}, Month={month}")
                    break
                month_end = next_month - datetime.timedelta(days=1)
                if month_end > end_date:
                    month_end = end_date
                logging.debug(f"Processing month range: {month_start} to {month_end}")
                process_date_range(query, month_start, month_end, processed_ids, all_ids)
                current_date = next_month  # Update current_date to the first day of the next month
        
        elif delta_days > 0:  # Split by day
            logging.info(f"Splitting {mindate} to {maxdate} by day.")
            for n in range(delta_days + 1):
                single_day = start_date + datetime.timedelta(n)
                logging.debug(f"Processing single day: {single_day}")
                process_date_range(query, single_day, single_day, processed_ids, all_ids)
        
        else:
            logging.warning(f"Single day {mindate} has more than 10,000 results, which is unexpected.")

# Main script execution
if __name__ == "__main__":
    import os
    import json
    import logging

    # Set up logging
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

    # Read your search query from a file or define it directly
    with open('search_strategy.txt', 'r') as f:
        query = f.read().strip()

    # Load processed IDs from existing results file
    processed_ids = load_processed_ids('pubmed_results_complete.csv')

    # Load existing IDs from 'pubmed_ids.json' or initialize an empty set
    if os.path.exists('pubmed_ids.json'):
        with open('pubmed_ids.json', 'r') as f:
            all_ids = set(json.load(f))
            logging.info(f"Loaded {len(all_ids)} IDs from pubmed_ids.json")
    else:
        all_ids = set()
        logging.info("pubmed_ids.json not found. Starting with an empty ID set.")

    # Define start and end dates
    start_year = 2011 # Replace with your desired start year
    start_date = datetime.date(start_year, 1, 1)
    end_date = datetime.date.today()  # Up to today's date

    # Start processing date ranges
    process_date_range(query, start_date, end_date, processed_ids, all_ids)

    # Save the updated list of IDs to 'pubmed_ids.json'
    with open('pubmed_ids.json', 'w') as f:
        json.dump(list(all_ids), f)
        logging.info(f"Updated pubmed_ids.json with {len(all_ids)} IDs.")

    logging.info("Data fetching complete.")

KeyboardInterrupt: 